In [ ]:
import numpy as np

# --- Dataset -----------------------------------------------------------
# Features (one-hot encoded): [Brown Cap, Tapering Stalk, Solitary]
# Label: 1 = edible, 0 = poisonous
X_train = np.array([[1, 1, 1],
                     [1, 0, 1],
                     [1, 0, 0],
                     [1, 0, 0],
                     [1, 1, 1],
                     [0, 1, 1],
                     [0, 0, 0],
                     [1, 0, 1],
                     [0, 1, 0],
                     [1, 0, 0]])
y_train = np.array([1, 1, 0, 0, 1, 0, 0, 1, 1, 0])
feature_names = ["Brown Cap", "Tapering Stalk Shape", "Solitary"]


def compute_entropy(y):
    # Impurity of a node: 0 = pure, 1 = maximally mixed (50/50)
    entropy = 0.
    if len(y) != 0:
        p1 = len(y[y == 1]) / len(y)              # fraction labeled edible (1)
        if p1 != 0 and p1 != 1:
            entropy = -p1 * np.log2(p1) - (1 - p1) * np.log2(1 - p1)
        else:
            entropy = 0.
    return entropy


def split_dataset(X, node_indices, feature):
    # Sends each index into left (feature==1) or right (feature==0)
    left_indices = []
    right_indices = []
    for i in node_indices:
        if X[i][feature] == 1:
            left_indices.append(i)
        else:
            right_indices.append(i)
    return left_indices, right_indices


def compute_information_gain(X, y, node_indices, feature):
    # How much splitting on `feature` reduces entropy vs. the parent node
    left_indices, right_indices = split_dataset(X, node_indices, feature)

    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]

    node_entropy = compute_entropy(y_node)         # impurity before splitting
    left_entropy = compute_entropy(y_left)          # impurity of left branch
    right_entropy = compute_entropy(y_right)        # impurity of right branch

    w_left = len(X_left) / len(X_node)               # fraction of examples going left
    w_right = len(X_right) / len(X_node)             # fraction of examples going right

    weighted_entropy = w_left * left_entropy + w_right * right_entropy
    information_gain = node_entropy - weighted_entropy
    return information_gain


def get_best_split(X, y, node_indices):
    # Tries every feature, returns the index of the one with highest gain
    num_features = X.shape[1]
    best_feature = -1
    max_info_gain = 0
    for feature in range(num_features):
        info_gain = compute_information_gain(X, y, node_indices, feature)
        if info_gain > max_info_gain:
            max_info_gain = info_gain
            best_feature = feature
    return best_feature


tree = []  # collects (left_indices, right_indices, feature) for each internal node


def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth):
    # Recursively splits until max_depth is reached; prints the tree as it builds
    if current_depth == max_depth:
        formatting = " " * current_depth + "-" * current_depth
        print(formatting, "%s leaf node with indices" % branch_name, node_indices)
        return

    best_feature = get_best_split(X, y, node_indices)   # pick the winning feature
    formatting = "-" * current_depth
    print("%s Depth %d, %s: Split on feature: %d (%s)" %
          (formatting, current_depth, branch_name, best_feature, feature_names[best_feature]))

    left_indices, right_indices = split_dataset(X, node_indices, best_feature)
    tree.append((left_indices, right_indices, best_feature))

    build_tree_recursive(X, y, left_indices, "Left", max_depth, current_depth + 1)
    build_tree_recursive(X, y, right_indices, "Right", max_depth, current_depth + 1)


if __name__ == "__main__":
    root_indices = list(range(10))

    print("Entropy at root node:", compute_entropy(y_train))   # should be 1.0 (5 edible / 5 poisonous)
    print()

    for f, name in enumerate(feature_names):
        gain = compute_information_gain(X_train, y_train, root_indices, f)
        print(f"Information gain splitting on {name}: {gain:.4f}")
    print()

    best = get_best_split(X_train, y_train, root_indices)
    print(f"Best feature to split root on: {best} ({feature_names[best]})")
    print()

    print("Building tree (max_depth=2):")
    build_tree_recursive(X_train, y_train, root_indices, "Root", max_depth=2, current_depth=0)